In [0]:
%sql
CREATE OR REPLACE VIEW dbdemos_vishesh.bharat_bricks.iitb_subreddit_metrics
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1

  source: >
    SELECT
      p.post_id,
      p.title,
      p.body,
      p.author,
      p.created_at,
      p.score        AS post_score,
      p.upvote_ratio,
      p.num_comments,
      p.flair,
      p.author_flair_text,
      p.is_self,
      p.is_video,
      p.is_original_content,
      p.spoiler,
      p.locked,
      p.domain,
      p.num_crossposts,
      p.subreddit_subscribers,
      p.content_type,
      p.permalink,
      COALESCE(c.total_comment_score, 0)  AS total_comment_score,
      COALESCE(c.avg_comment_score,  0.0) AS avg_comment_score,
      COALESCE(c.max_thread_depth,   0)   AS max_thread_depth,
      COALESCE(c.top_level_comments, 0)   AS top_level_comments,
      COALESCE(c.op_replies,         0)   AS op_replies,
      COALESCE(c.unique_commenters,  0)   AS unique_commenters,
      COALESCE(c.total_awards,       0)   AS total_awards,
      COALESCE(c.edited_comments,    0)   AS edited_comments,
      COALESCE(c.mod_actions,        0)   AS mod_actions
    FROM dbdemos_vishesh.bharat_bricks.gold_posts p
    LEFT JOIN (
      SELECT
        post_id,
        SUM(score)                                                    AS total_comment_score,
        AVG(CAST(score AS DOUBLE))                                    AS avg_comment_score,
        MAX(depth)                                                    AS max_thread_depth,
        SUM(CASE WHEN depth = 0 THEN 1 ELSE 0 END)                   AS top_level_comments,
        SUM(CASE WHEN is_submitter THEN 1 ELSE 0 END)                AS op_replies,
        COUNT(DISTINCT author)                                        AS unique_commenters,
        SUM(total_awards_received)                                    AS total_awards,
        SUM(CASE WHEN edited != 'false' THEN 1 ELSE 0 END)           AS edited_comments,
        SUM(CASE WHEN distinguished = 'moderator' THEN 1 ELSE 0 END) AS mod_actions
      FROM dbdemos_vishesh.bharat_bricks.gold_comments
      GROUP BY post_id
    ) c ON p.post_id = c.post_id

  comment: >-
    Unified metric view for the r/iitbombay subreddit — IIT Bombay campus life,
    academics, placements, hostel culture, politics, and community discussions.
    Posts are the primary grain, enriched with pre-aggregated comment statistics.
    Use for campus sentiment, engagement trends, content analysis, and community
    health dashboards via AI/BI, Genie, or RAG agents.

  # ── Dimensions ────────────────────────────────────────────────────────
  dimensions:

    # ── Time ──
    - name: Post Date
      expr: DATE(created_at)
      display_name: Post Date
      comment: Date (UTC) the post was submitted to r/iitbombay.
      format:
        type: date
        date_format: year_month_day
      synonyms:
        - date
        - day
        - submission date
        - posted on

    - name: Post Month
      expr: DATE_TRUNC('MONTH', created_at)
      display_name: Post Month
      comment: Calendar month when the post was submitted.
      format:
        type: date
        date_format: locale_short_month
      synonyms:
        - month
        - monthly

    - name: Post Quarter
      expr: DATE_TRUNC('QUARTER', created_at)
      display_name: Post Quarter
      comment: Calendar quarter when the post was submitted.
      format:
        type: date
        date_format: year_month_day
      synonyms:
        - quarter
        - quarterly

    - name: Post Year
      expr: YEAR(created_at)
      display_name: Year
      comment: Year the post was submitted.
      synonyms:
        - year
        - yearly
        - annual

    - name: Day of Week
      expr: DATE_FORMAT(created_at, 'EEEE')
      display_name: Day of Week
      comment: Weekday name (Monday–Sunday) when the post was submitted (UTC).
      synonyms:
        - weekday
        - day name

    - name: Hour of Day
      expr: HOUR(created_at)
      display_name: Hour of Day (UTC)
      comment: Hour (0–23 UTC) when the post was submitted. Useful for finding peak activity windows.
      synonyms:
        - time of day
        - posting hour
        - hour

    - name: Academic Term
      expr: |-
        CASE
          WHEN MONTH(created_at) BETWEEN 1 AND 4  THEN 'Spring Semester'
          WHEN MONTH(created_at) BETWEEN 5 AND 7  THEN 'Summer Break'
          WHEN MONTH(created_at) BETWEEN 8 AND 12 THEN 'Autumn Semester'
        END
      display_name: Academic Term
      comment: >-
        Approximate IIT Bombay academic calendar mapping — Spring (Jan–Apr),
        Summer break (May–Jul), Autumn (Aug–Dec).
      synonyms:
        - semester
        - term
        - academic period
        - sem

    # ── Content ──
    - name: Flair
      expr: COALESCE(flair, 'Untagged')
      display_name: Post Flair
      comment: >-
        Subreddit category tag. Values: Question, Other, Tech, Survey, Acads,
        IIT Selection, Cult, Sports, Ask me anything!, Polt, or Untagged when
        the author did not pick a flair.
      synonyms:
        - category
        - topic
        - tag
        - post type
        - flair tag

    - name: Content Type
      expr: content_type
      display_name: Content Type
      comment: >-
        Media format of the post — text (self-post), image, gallery
        (multiple images), video, or link (external URL).
      synonyms:
        - media type
        - post format
        - type of content

    - name: Domain
      expr: domain
      display_name: Content Domain
      comment: >-
        Source domain of the post. self.iitbombay for text posts, i.redd.it
        for images, v.redd.it for Reddit-hosted videos, reddit.com for
        crossposts, or external sites like youtube.com.
      synonyms:
        - source
        - link domain
        - website

    # ── Author / Community ──
    - name: Author
      expr: author
      display_name: Author
      comment: Reddit username of the post author.
      synonyms:
        - poster
        - user
        - username
        - OP
        - redditor

    - name: Author Affiliation
      expr: COALESCE(author_flair_text, 'Unknown')
      display_name: Author Affiliation
      comment: >-
        Self-declared badge next to the username. Typically a department
        (Elec, Mech, CS, Chem, Energy), hostel (H2 Wild Ones, H5 Penthouse,
        H9 Nawaab, H4 Madhouse, H1 Queen), student body or club (WnCC,
        Krittika, Mood Indigo), or Alum for alumni.
      synonyms:
        - department
        - hostel
        - affiliation
        - branch
        - author flair
        - wing
        - club

    - name: Affiliation Category
      expr: |-
        CASE
          WHEN author_flair_text IN ('CS','Elec','Mech','Chem','Energy')                             THEN 'Department'
          WHEN author_flair_text LIKE 'H%'                                                           THEN 'Hostel'
          WHEN author_flair_text IN ('WnCC','Krittika','Mood Indigo')                                THEN 'Student Body / Club'
          WHEN author_flair_text = 'Alum'                                                            THEN 'Alumni'
          ELSE 'Unknown'
        END
      display_name: Affiliation Category
      comment: >-
        Broad grouping of author flair into Department, Hostel, Student Body / Club,
        Alumni, or Unknown.
      synonyms:
        - flair group
        - affiliation type
        - author type

    # ── Post flags ──
    - name: Is Text Post
      expr: is_self
      display_name: Is Text Post
      comment: Whether this is a self/text post (true) vs link/media post (false).
      synonyms:
        - self post
        - text only

    - name: Is Video
      expr: is_video
      display_name: Is Video
      comment: Whether the post contains a Reddit-hosted video.
      synonyms:
        - video post
        - has video

    - name: Is Locked
      expr: locked
      display_name: Is Locked
      comment: Whether moderators locked the post, preventing new comments.
      synonyms:
        - locked
        - mod locked
        - closed

    - name: Is Original Content
      expr: is_original_content
      display_name: Is Original Content
      comment: Whether the author marked the post as Original Content (OC).
      synonyms:
        - OC
        - original

    - name: Post Title
      expr: title
      display_name: Post Title
      comment: Title / headline of the Reddit post as written by the author.
      synonyms:
        - headline
        - subject
        - post name

    - name: Post Link
      expr: CONCAT('https://www.reddit.com', permalink)
      display_name: Post URL
      comment: Full URL to the post on Reddit.
      synonyms:
        - url
        - reddit link
        - post url

  # ── Measures ──────────────────────────────────────────────────────────
  measures:

    # ── Volume ──
    - name: Total Posts
      expr: COUNT(1)
      display_name: Total Posts
      comment: Total number of posts submitted to r/iitbombay.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - post count
        - number of posts
        - submissions

    - name: Total Comments
      expr: SUM(num_comments)
      display_name: Total Comments
      comment: Sum of comment counts across all posts.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - comment count
        - replies
        - responses
        - number of comments

    - name: Unique Authors
      expr: COUNT(DISTINCT author)
      display_name: Unique Authors
      comment: Number of distinct Reddit users who authored posts.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - unique posters
        - distinct users
        - active posters

    # ── Scoring / Sentiment ──
    - name: Avg Post Score
      expr: AVG(post_score)
      display_name: Avg Post Score
      comment: Mean net vote score per post (upvotes minus downvotes). Reddit floors post scores at 0.
      format:
        type: number
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - average score
        - mean score
        - avg karma

    - name: Median Post Score
      expr: PERCENTILE(post_score, 0.5)
      display_name: Median Post Score
      comment: Median net vote score — more robust than the average in a heavy-tailed distribution.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - median score
        - typical score

    - name: Max Post Score
      expr: MAX(post_score)
      display_name: Max Post Score
      comment: Highest single-post score in the selection.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - top score
        - highest score
        - peak score

    - name: Total Post Score
      expr: SUM(post_score)
      display_name: Total Post Score
      comment: Sum of net vote scores across all posts.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - total karma
        - cumulative score

    - name: Avg Upvote Ratio
      expr: AVG(upvote_ratio)
      display_name: Avg Upvote Ratio
      comment: >-
        Average fraction of upvotes (0.0–1.0). Higher values mean more
        positive community reception.
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - approval rate
        - upvote percentage
        - sentiment ratio
        - positivity

    # ── Engagement ──
    - name: Avg Comments per Post
      expr: AVG(num_comments)
      display_name: Avg Comments per Post
      comment: Mean number of comments per post — a proxy for discussion intensity.
      format:
        type: number
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - comments per post
        - avg replies
        - discussion rate

    - name: Total Engagement Score
      expr: SUM(post_score) + SUM(num_comments)
      display_name: Total Engagement Score
      comment: Combined post votes + comment count as a single engagement proxy.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - engagement
        - activity score
        - total activity

    - name: High Engagement Posts
      expr: |-
        SUM(CASE
          WHEN num_comments >= 20 OR post_score >= 100 THEN 1
          ELSE 0
        END)
      display_name: High Engagement Posts
      comment: >-
        Posts with 20 + comments or 100 + score — viral or deeply discussed
        threads on campus.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - viral posts
        - trending posts
        - hot posts
        - popular posts

    - name: High Engagement Rate
      expr: |-
        SUM(CASE WHEN num_comments >= 20 OR post_score >= 100 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(1), 0)
      display_name: High Engagement Rate
      comment: Fraction of posts classified as high-engagement.
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - viral rate
        - trending rate

    # ── Comment quality (from pre-aggregated stats) ──
    - name: Avg Comment Score
      expr: AVG(avg_comment_score)
      display_name: Avg Comment Score
      comment: Mean per-post average comment score — indicates quality of discussion.
      format:
        type: number
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - comment quality
        - reply score

    - name: Avg Unique Commenters
      expr: AVG(unique_commenters)
      display_name: Avg Unique Commenters per Post
      comment: Average number of distinct users participating in each thread.
      format:
        type: number
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - commenters per post
        - discussion breadth
        - participation

    - name: Avg Thread Depth
      expr: AVG(max_thread_depth)
      display_name: Avg Thread Depth
      comment: >-
        Average deepest nesting level per thread. Higher depth = sustained
        back-and-forth conversation.
      format:
        type: number
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - conversation depth
        - nesting level
        - thread length

    - name: OP Engagement Rate
      expr: SUM(op_replies) / NULLIF(SUM(num_comments), 0)
      display_name: OP Engagement Rate
      comment: Fraction of total comments that come from the original poster. Higher = more author involvement.
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - op reply rate
        - author participation

    # ── Moderation / Community Health ──
    - name: Locked Post Count
      expr: SUM(CASE WHEN locked THEN 1 ELSE 0 END)
      display_name: Locked Post Count
      comment: Number of posts locked by moderators.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - locked posts
        - mod locks

    - name: Locked Post Rate
      expr: SUM(CASE WHEN locked THEN 1 ELSE 0 END) / NULLIF(COUNT(1), 0)
      display_name: Locked Post Rate
      comment: Fraction of posts locked by moderators — a signal of controversial content.
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - moderation rate
        - lock rate

    - name: Mod Action Count
      expr: SUM(mod_actions)
      display_name: Mod Action Count
      comment: Total moderator-distinguished comments across posts.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - moderator interventions
        - mod comments

    - name: Controversial Post Rate
      expr: |-
        SUM(CASE WHEN upvote_ratio < 0.6 AND num_comments >= 5 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(1), 0)
      display_name: Controversial Post Rate
      comment: >-
        Fraction of posts with <60 % upvote ratio and at least 5 comments —
        divisive campus topics.
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - controversy rate
        - divisive post rate

    # ── Content mix ──
    - name: Text Post Ratio
      expr: SUM(CASE WHEN is_self THEN 1 ELSE 0 END) / NULLIF(COUNT(1), 0)
      display_name: Text Post Ratio
      comment: Fraction of posts that are text/self posts vs media/link posts.
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - self post ratio
        - text percentage

    - name: Total Crossposts
      expr: SUM(num_crossposts)
      display_name: Total Crossposts
      comment: Number of times posts were shared to other subreddits — indicates external reach.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - shares
        - crossposts

    - name: Total Awards
      expr: SUM(total_awards)
      display_name: Total Awards
      comment: Total Reddit awards (Gold, Silver, community) received by comments.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - awards
        - gilded
        - rewards

    # ── Window: trailing activity ──
    - name: Daily Posts
      expr: COUNT(1)
      window:
        - order: Post Date
          range: current
          semiadditive: last
      display_name: Daily Posts
      comment: Post count for each calendar day — used as a building block for trailing windows.

    - name: T7D Posts
      expr: MEASURE(`Daily Posts`)
      window:
        - order: Post Date
          range: trailing 7 day
          semiadditive: last
      display_name: Trailing 7-Day Posts
      comment: Rolling 7-day post count — smooths daily noise to reveal weekly momentum.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - weekly posts
        - 7 day rolling
        - last 7 days

    - name: T30D Posts
      expr: MEASURE(`Daily Posts`)
      window:
        - order: Post Date
          range: trailing 30 day
          semiadditive: last
      display_name: Trailing 30-Day Posts
      comment: Rolling 30-day post count — captures monthly activity trends.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - monthly posts
        - 30 day rolling
        - last 30 days

    - name: Cumulative Posts
      expr: COUNT(1)
      window:
        - order: Post Date
          range: cumulative
          semiadditive: last
      display_name: Cumulative Posts
      comment: Running total of posts since the first submission.
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - running total
        - all time posts
$$